# 🥬 Vegetable Freshness — Train on Google Colab

Notebook **chạy mượt** trên Colab Free (T4) hoặc Pro (V100/A100). Tối ưu sẵn:
- ✅ Kiểm tra GPU + cảnh báo nếu chưa bật
- ✅ Mount Google Drive (giữ dataset & checkpoint qua các phiên)
- ✅ Cài đúng dependency, KHÔNG đụng tensorflow của Colab
- ✅ Mixed Precision (FP16) + XLA → train nhanh ~1.5–2×
- ✅ TensorBoard inline
- ✅ Auto-resume nếu checkpoint đã tồn tại trong Drive

**Trước khi chạy**: `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L || echo '⚠️  Chưa bật GPU! Vào Runtime → Change runtime type → GPU'

## 2. Mount Google Drive

Mọi thứ sẽ lưu vào `/content/drive/MyDrive/veggie_project/` để KHÔNG mất khi Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/veggie_project')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ('dataset', 'checkpoints', 'logs', 'results'):
    (DRIVE_ROOT / sub).mkdir(exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 3. Lấy source code

**Cách A — Clone từ GitHub (khuyến nghị)**: thay `<your-repo>` bên dưới.

**Cách B — Upload thủ công**: dùng panel file trái, kéo thả thư mục `project/` vào `/content/`.

In [ ]:
# Cách A: clone từ GitHub
%cd /content
# !git clone https://github.com/<your-user>/<your-repo>.git project

# Cách B: nếu đã upload thư mục project/ qua panel
%cd /content/project
!ls

## 4. Cài dependencies (chỉ cài thêm cái Colab thiếu)

Colab đã có sẵn `tensorflow`, `numpy`, `matplotlib`, `pandas`, `seaborn`, `scikit-learn`, `Pillow`, `opencv-python`. Chỉ cài bổ sung.

In [ ]:
!pip install -q icrawler imagehash tqdm kagglehub
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPU       :', tf.config.list_physical_devices('GPU'))

## 5. Symlink Drive ↔ project (tránh sao chép data 10GB)

Trỏ thư mục `dataset/`, `checkpoints/`, `logs/`, `results/` của project sang Drive. Mọi thay đổi tự động lưu vào Drive.

In [ ]:
import os, pathlib, shutil
PROJ = pathlib.Path('/content/project')
for sub in ('dataset', 'checkpoints', 'logs', 'results'):
    target = DRIVE_ROOT / sub
    link   = PROJ / sub
    if link.exists() and not link.is_symlink():
        shutil.rmtree(link)
    if not link.exists():
        os.symlink(target, link)
    print(f'  {link}  →  {target}')

## 6. Lấy dataset Freshness44 từ Kaggle (KHUYẾN NGHỊ)

Dataset [Freshness44](https://www.kaggle.com/datasets/siavash93/freshness44) — 53.616 ảnh, 22 loại rau/quả, đã clean sẵn (~6.7 GB).

Lần đầu sẽ mở browser yêu cầu đăng nhập Kaggle. Có thể thay bằng `kaggle.json` upload thủ công.

In [ ]:
%cd /content/project
# Khuyên: --max-per-class 8000 để giữ data ~16k ảnh, train nhanh hơn
!python tools/prepare_freshness44.py --max-per-class 8000

## 6b. (Tuỳ chọn) Thay vì Freshness44, có thể tự crawl

In [ ]:
%cd /content/project
# Bỏ comment để chạy crawl thay vì dùng Kaggle
# !python crawler/crawl_images.py --target 10000 --engines google,bing,baidu

## 7. Tiền xử lý + split (resize 224, train/valid/test 70/15/15)

In [ ]:
%cd /content/project
# Bỏ comment để chạy tiền xử lý
# !python preprocessing/preprocess.py --src dataset/raw --dst dataset --img-size 224

## 8. TensorBoard inline (theo dõi training real-time)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/project/logs

## 9. Train — MobileNetV2 (nhanh, nhẹ, hợp Colab Free)

Tối ưu Colab:
- `--mixed-precision` → FP16, tăng tốc 1.5–2× trên T4/V100/A100
- `--xla` → JIT compile, +10–20% tốc độ
- `--batch-size 64` (T4 16GB VRAM gánh tốt với MobileNet)

In [ ]:
%cd /content/project
!python models/train.py \
    --model mobilenet \
    --epochs 30 \
    --ft-epochs 15 \
    --batch-size 64 \
    --mixed-precision \
    --xla

## 10. Train — ResNet50 (nặng hơn, batch nhỏ lại nếu OOM)

In [ ]:
%cd /content/project
!python models/train.py \
    --model resnet \
    --epochs 30 \
    --ft-epochs 15 \
    --batch-size 32 \
    --mixed-precision \
    --xla

## 11. Đánh giá + so sánh

In [ ]:
%cd /content/project
!python evaluation/evaluate.py --model checkpoints/mobilenet_best.keras
!python evaluation/evaluate.py --model checkpoints/resnet_best.keras
!python evaluation/plots.py

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/project/results/compare_*.png') +
                glob.glob('/content/project/results/*_confusion_matrix.png')):
    print(p); display(Image(p))

## 12. Test predict

In [ ]:
%cd /content/project
!python app/predict.py --image dataset/test/fresh --model checkpoints/mobilenet_best.keras --show
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/project/results/predictions/*.png'))[:6]:
    display(Image(p))

## 13. Mẹo giữ phiên Colab Free không bị disconnect

Mở **Console** trình duyệt (F12) và dán đoạn JS dưới — sẽ tự click nhẹ mỗi 60s:

```js
function ClickConnect(){
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ClickConnect, 60000);
```

**Khuyến nghị quan trọng**:
- Lưu mọi thứ vào Drive (đã làm ở bước 5) để không mất khi runtime hết hạn (12h Free / 24h Pro).
- Nếu OOM: giảm `--batch-size` xuống 16, hoặc bỏ `--mixed-precision`.
- Sau khi train xong, file model `.keras` đã nằm sẵn trong `/content/drive/MyDrive/veggie_project/checkpoints/` — tải về máy bằng panel file.